In [1]:
# =========================
# CELL 1 — Imports & Setup
# =========================
import os
import json
import math
import copy
import random
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import Levenshtein as Lev

pd.set_option("display.max_colwidth", None)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print(torch.__version__)
print(torch.cuda.is_available())

Using device: cuda
2.5.1+cu121
True


In [2]:
# =========================
# CELL 2 — Load Metadata JSON
# =========================
videos = os.path.join("..", "videos_all_languages.json")

with open(videos, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data["videos"])

print(df.columns)
print(df.head())

Index(['id', 'language', 'Sign', 'filepath', 'HamNoSys', 'concept_url',
       'HandLandmark', 'annotated_video'],
      dtype='object')
      id language      Sign  \
0  BSL_1      BSL     April   
1  BSL_2      BSL    Athens   
2  BSL_3      BSL    August   
3  BSL_4      BSL    Berlin   
4  BSL_5      BSL  February   

                                            filepath  \
0     /ephemeral/Project/Data/External/BSL/April.mp4   
1    /ephemeral/Project/Data/External/BSL/Athens.mp4   
2    /ephemeral/Project/Data/External/BSL/August.mp4   
3    /ephemeral/Project/Data/External/BSL/Berlin.mp4   
4  /ephemeral/Project/Data/External/BSL/February.mp4   

                                          HamNoSys  \
0     
1                 
2         
3                                         
4                             

              

In [3]:
# =========================
# CELL 3 — Get Landmark Paths
# =========================
Hand_Landmarks_paths = df["HandLandmark"]
Hand_Landmarks = [p for p in Hand_Landmarks_paths if os.path.exists(p)]

print("Valid landmark files:", len(Hand_Landmarks))

Valid landmark files: 3037


In [4]:
# =========================
# CELL 4 — Utility Functions
# =========================
def fix_mojibake(name):
    try:
        name = name.encode("latin1").decode("utf8")
    except:
        pass

    replacements = {
        "ÃŸ": "ß", "Ã„": "Ä", "Ã–": "Ö", "Ãœ": "Ü",
        "Ã¤": "ä", "Ã¶": "ö", "Ã¼": "ü",
        "Ã©": "é", "Ã¨": "è", "Ãª": "ê", "Ã«": "ë",
        "Ã´": "ô", "Ã¢": "â", "Ã¹": "ù", "Ã»": "û",
        "Ã§": "ç",
    }

    for bad, good in replacements.items():
        name = name.replace(bad, good)

    return unicodedata.normalize("NFC", name)


def normalize_frame_features(frame_features):
    arr = np.array(frame_features, dtype=np.float32).reshape(-1, 3)

    valid = np.any(arr != 0, axis=1)

    if valid.sum() > 1:
        center = arr[valid].mean(axis=0)
        arr[valid] -= center

        scale = np.std(arr[valid])

        if np.isfinite(scale) and scale > 1e-6:
            arr[valid] /= scale
        else:
            arr[:] = 0

    return arr.reshape(-1).tolist()

In [5]:
# =========================
# CELL 5 — Landmark Index Lists
# =========================
REDUCED_FACE_LANDMARKS = (
    list(range(70, 103)) +
    list(range(0, 17)) +
    list(range(61, 88)) +
    list(range(152, 172))
)

HAND_LANDMARK_IDS = list(range(21))
POSE_LANDMARK_IDS = list(range(33))

In [6]:
# =========================
# CELL 6 — Convert Frames to Sequence
# =========================
def convert_frames_to_sequence(frames):
    rows = []

    for frame in frames:
        feats = []

        left = {
            lm["id"]: lm
            for hand in frame["hands"]
            if hand["handedness"] == "Left"
            for lm in hand["landmarks"]
        }

        right = {
            lm["id"]: lm
            for hand in frame["hands"]
            if hand["handedness"] == "Right"
            for lm in hand["landmarks"]
        }

        for i in HAND_LANDMARK_IDS:
            lm = left.get(i, {"x":0,"y":0,"z":0})
            feats.extend([lm["x"], lm["y"], lm["z"]])

        for i in HAND_LANDMARK_IDS:
            lm = right.get(i, {"x":0,"y":0,"z":0})
            feats.extend([lm["x"], lm["y"], lm["z"]])

        pose_dict = {
            lm["id"]: lm
            for pose in frame["pose"]
            for lm in pose["landmarks"]
        }

        for i in POSE_LANDMARK_IDS:
            lm = pose_dict.get(i, {"x":0,"y":0,"z":0})
            feats.extend([lm["x"], lm["y"], lm["z"]])

        face_dict = {
            lm["id"]: lm
            for face in frame["face"]
            for lm in face["landmarks"]
        }

        for i in REDUCED_FACE_LANDMARKS:
            lm = face_dict.get(i, {"x":0,"y":0,"z":0})
            feats.extend([lm["x"], lm["y"], lm["z"]])

        feats = normalize_frame_features(feats)
        rows.append(np.array(feats, dtype=np.float32))

    return np.array(rows, dtype=np.float32)

In [ ]:
# =========================
# CELL 7 — Convert JSON Files to .npy
# =========================
os.makedirs("processed_sequences", exist_ok=True)

for path in Hand_Landmarks:
    with open(path, "r") as f:
        data = json.load(f)

    raw = data["video_path"]

    video_id = fix_mojibake(
        os.path.basename(raw)
        .replace(".mp4", "")
        .replace(".MOV", "")
        .replace("_cropped", "")
    )

    seq = convert_frames_to_sequence(data["frames"])

    np.save(f"processed_sequences/{video_id}.npy", seq)

print("Saved processed sequences.")

In [ ]:
# =========================
# CELL 8 — Build Label Dictionary
# =========================
df["filepath"] = df["filepath"].apply(
    lambda p: fix_mojibake(
        os.path.basename(p)
        .replace(".mp4", "")
        .replace(".MOV", "")
    )
)

label_dict = dict(zip(df["filepath"], df["HamNoSys"]))
print("Labels:", len(label_dict))

Labels: 2997


In [ ]:
# =========================
# CELL 9 — Load X / y
# =========================
X = []
y = []

for seq_file in os.listdir("processed_sequences"):
    vid = seq_file.replace(".npy", "")
    clean_vid = fix_mojibake(vid)

    if clean_vid in label_dict:
        X.append(np.load(f"processed_sequences/{seq_file}"))
        y.append(label_dict[clean_vid])

print("Samples:", len(X))

Samples: 3055


In [ ]:
# =========================
# CELL 10 — Build Vocabulary
# =========================
all_tokens = sorted({c for seq in y for c in seq})

token_to_id = {t:i+3 for i,t in enumerate(all_tokens)}
token_to_id["<PAD>"] = 0
token_to_id["<SOS>"] = 1
token_to_id["<EOS>"] = 2

id_to_token = {v:k for k,v in token_to_id.items()}

pad_id = 0
sos_id = 1
eos_id = 2

vocab_size = len(token_to_id)
print("Vocab size:", vocab_size)

Vocab size: 188


In [ ]:
# =========================
# CELL 11 — Encode Labels
# =========================
y_sequences = []

for seq in y:
    encoded = [sos_id] + [token_to_id[c] for c in seq] + [eos_id]
    y_sequences.append(encoded)

In [ ]:
# =========================
# CELL 12 — Train/Val/Test Split
# =========================
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_sequences, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(len(X_train), len(X_val), len(X_test))

2138 458 459


In [ ]:
# =========================
# CELL 13 — Padding Functions
# =========================
def pad_sequences_X(seqs, max_len):

    num_features = seqs[0].shape[1]

    X = torch.zeros((len(seqs), max_len, num_features), dtype=torch.float32)
    mask = torch.zeros((len(seqs), max_len), dtype=torch.bool)

    for i, seq in enumerate(seqs):

        seq = np.asarray(seq, dtype=np.float32)  # ✅ force clean dtype

        L = min(len(seq), max_len)

        X[i, :L] = torch.from_numpy(seq[:L])     # ✅ faster + safer
        mask[i, :L] = 1

    return X, mask


def pad_sequences_y(labels, max_len):
    Y = torch.full((len(labels), max_len), pad_id)

    for i, seq in enumerate(labels):
        L = min(len(seq), max_len)
        Y[i,:L] = torch.tensor(seq[:L])

    return Y.long()

In [ ]:
import numpy as np
import torch

print(np.__version__)
print(torch.__version__)
print(torch.from_numpy(np.array([1,2,3])))

1.26.4
2.5.1+cu121
tensor([1, 2, 3])


In [ ]:
# =========================
# CELL 14 — Create Tensors
# =========================
max_len_X = max(len(s) for s in X_train + X_val + X_test)
max_len_y = max(len(s) for s in y_train + y_val + y_test)

X_train_tensor, X_train_mask = pad_sequences_X(X_train, max_len_X)
X_val_tensor, X_val_mask = pad_sequences_X(X_val, max_len_X)
X_test_tensor, X_test_mask = pad_sequences_X(X_test, max_len_X)

y_train_tensor = pad_sequences_y(y_train, max_len_y)
y_val_tensor = pad_sequences_y(y_val, max_len_y)
y_test_tensor = pad_sequences_y(y_test, max_len_y)

In [ ]:
# =========================
# CELL 15 — DataLoaders
# =========================
train_dataset = TensorDataset(X_train_tensor, X_train_mask, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor, X_val_mask, y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor, X_test_mask, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

print("Ready.")

Ready.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

import lightning as L
import torchvision
import torchaudio

import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import math
import copy
import random
import os

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import Levenshtein as Lev

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Using device: cuda
True
1
NVIDIA H100 PCIe


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_sequences, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

In [ ]:
def temporal_subsample(seq, rates=[2,3], min_frames=20):
    augmented = []

    for r in rates:
        new_seq = seq[::r]
        if new_seq.shape[0] >= min_frames:
            augmented.append(new_seq)

    return augmented


def random_frame_drop(seq, drop_prob=0.05, min_frames=20):
    mask = np.random.rand(seq.shape[0]) > drop_prob
    new_seq = seq[mask]

    if new_seq.shape[0] >= min_frames:
        return new_seq

    return None

In [ ]:
X_aug = []
y_aug = []

MIN_FRAMES = 20

for x_seq, y_seq in zip(X_train, y_train):

    if x_seq.shape[0] >= MIN_FRAMES:
        X_aug.append(x_seq)
        y_aug.append(y_seq)

    for new_seq in temporal_subsample(x_seq):
        X_aug.append(new_seq)
        y_aug.append(y_seq)

    dropped = random_frame_drop(x_seq)

    if dropped is not None:
        X_aug.append(dropped)
        y_aug.append(y_seq)

print("Augmented samples:", len(X_aug))

Augmented samples: 7458


In [ ]:
def pad_sequences_X(seqs, max_len):
    num_features = seqs[0].shape[1]

    X_pad = torch.zeros((len(seqs), max_len, num_features), dtype=torch.float32)
    mask = torch.zeros((len(seqs), max_len), dtype=torch.bool)

    for i, seq in enumerate(seqs):
        length = min(seq.shape[0], max_len)

        X_pad[i, :length] = torch.tensor(seq[:length], dtype=torch.float32)
        mask[i, :length] = 1

    return X_pad, mask


def pad_sequences_y(labels, max_len, pad_token=0):
    Y = torch.full((len(labels), max_len), pad_token, dtype=torch.long)

    for i, seq in enumerate(labels):
        length = min(len(seq), max_len)
        Y[i, :length] = torch.tensor(seq[:length], dtype=torch.long)

    return Y

In [ ]:
max_len_X = max(seq.shape[0] for seq in X_aug + X_val + X_test)
max_len_y = max(len(seq) for seq in y_aug + y_val + y_test)

X_train_tensor, X_train_mask = pad_sequences_X(X_aug, max_len_X)
X_val_tensor, X_val_mask = pad_sequences_X(X_val, max_len_X)
X_test_tensor, X_test_mask = pad_sequences_X(X_test, max_len_X)

y_train_tensor = pad_sequences_y(y_aug, max_len_y)
y_val_tensor   = pad_sequences_y(y_val, max_len_y)
y_test_tensor  = pad_sequences_y(y_test, max_len_y)

In [ ]:
train_dataset = TensorDataset(
    X_train_tensor,
    X_train_mask,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    X_val_mask,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    X_test_mask,
    y_test_tensor
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class EncoderTransformer(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers=6,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, hidden_dim)

        self.pos_encoding = PositionalEncoding(hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, src_key_padding_mask=None):
        x = self.input_proj(x)
        x = self.pos_encoding(x)

        x = self.encoder(
            x,
            src_key_padding_mask=src_key_padding_mask
        )

        return self.norm(x)

In [ ]:
class CustomTransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.2):
        super().__init__()

        self.self_attn = nn.MultiheadAttention(
            d_model,
            nhead,
            dropout=dropout,
            batch_first=True
        )

        self.cross_attn = nn.MultiheadAttention(
            d_model,
            nhead,
            dropout=dropout,
            batch_first=True
        )

        self.linear1 = nn.Linear(d_model, d_model * 4)
        self.linear2 = nn.Linear(d_model * 4, d_model)

        self.dropout = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(
        self,
        tgt,
        memory,
        tgt_mask,
        tgt_key_padding_mask,
        memory_key_padding_mask
    ):

        # Self Attention
        _tgt, _ = self.self_attn(
            tgt,
            tgt,
            tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask
        )

        tgt = self.norm1(tgt + self.dropout1(_tgt))

        # Cross Attention
        _tgt, cross_attn_weights = self.cross_attn(
            tgt,
            memory,
            memory,
            key_padding_mask=memory_key_padding_mask,
            need_weights=True,
            average_attn_weights=False
        )

        tgt = self.norm2(tgt + self.dropout2(_tgt))

        # Feed Forward
        ff = self.linear2(
            self.dropout(
                F.relu(
                    self.linear1(tgt)
                )
            )
        )

        tgt = self.norm3(tgt + self.dropout3(ff))

        return tgt, cross_attn_weights

In [ ]:
class CustomTransformerDecoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        hidden_dim,
        num_layers=2,
        num_heads=4,
        dropout=0.15
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_dim)

        self.layers = nn.ModuleList([
            CustomTransformerDecoderLayer(
                hidden_dim,
                num_heads,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(hidden_dim, vocab_size)

        self.positional_encoding = PositionalEncoding(hidden_dim)

    def forward(
        self,
        tgt,
        memory,
        tgt_mask,
        tgt_key_padding_mask,
        memory_key_padding_mask
    ):

        x = self.embedding(tgt)

        x = x + self.positional_encoding.pe[:, :x.size(1)]

        cross_attn_all_layers = []

        for layer in self.layers:
            x, cross_attn = layer(
                x,
                memory,
                tgt_mask,
                tgt_key_padding_mask,
                memory_key_padding_mask
            )

            cross_attn_all_layers.append(cross_attn)

        logits = self.fc_out(x)

        return logits, cross_attn_all_layers

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(
        self,
        input_dim,
        vocab_size,
        hidden_dim=256
    ):
        super().__init__()

        self.encoder = EncoderTransformer(
            input_dim,
            hidden_dim
        )

        self.decoder = CustomTransformerDecoder(
            vocab_size,
            hidden_dim
        )

    def generate_square_subsequent_mask(self, size):
        return torch.triu(
            torch.ones(size, size),
            diagonal=1
        ).bool().to(device)

    def forward(self, src, src_mask, tgt):

        src = src.to(device)
        tgt = tgt.to(device)
        src_mask = src_mask.to(device)

        src_key_padding_mask = (src_mask == 0).bool()
        tgt_key_padding_mask = (tgt == pad_id).bool()

        memory = self.encoder(
            src,
            src_key_padding_mask=src_key_padding_mask
        )

        tgt_mask = self.generate_square_subsequent_mask(
            tgt.size(1)
        )

        logits, cross_attn = self.decoder(
            tgt=tgt,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        return logits, cross_attn

In [ ]:
criterion = nn.CrossEntropyLoss(
    ignore_index=pad_id,
    label_smoothing=0.1
)

In [ ]:
def compute_coverage_loss(cross_attn_layers):

    attn = cross_attn_layers[-1]      # final decoder layer
    attn = attn.mean(dim=1)          # average heads

    batch, tgt_len, src_len = attn.shape

    coverage = torch.zeros(
        batch,
        src_len,
        device=attn.device
    )

    cov_loss = 0.0

    for t in range(tgt_len):
        a_t = attn[:, t, :]
        cov_loss += torch.sum(
            torch.min(a_t, coverage)
        )
        coverage = coverage + a_t

    cov_loss = cov_loss / batch

    return cov_loss

In [ ]:
X_tensor = X_train_tensor
num_features = X_tensor.shape[2]

model = Seq2Seq(
    input_dim=num_features,
    vocab_size=vocab_size,
    hidden_dim=256
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
ema_model = copy.deepcopy(model)

ema_decay = 0.999


def update_ema(model, ema_model, decay=0.999):

    with torch.no_grad():

        for p, ema_p in zip(
            model.parameters(),
            ema_model.parameters()
        ):
            ema_p.data.mul_(decay).add_(
                p.data,
                alpha=1 - decay
            )

In [ ]:
def get_lr(
    epoch,
    base_lr=3e-4,
    warmup_epochs=3,
    max_epochs=100
):

    if epoch < warmup_epochs:
        return base_lr * (epoch / warmup_epochs)

    progress = (
        (epoch - warmup_epochs) /
        (max_epochs - warmup_epochs)
    )

    return base_lr * 0.5 * (
        1 + math.cos(math.pi * progress)
    )

In [ ]:
def token_accuracy(output, trg, pad_idx=0):

    pred = output.argmax(1)

    mask = trg != pad_idx

    correct = (pred == trg) & mask

    acc = correct.sum().item() / mask.sum().item()

    return acc * 100

In [ ]:
def clean(seq, sos_id, eos_id, pad_id):

    seq = [
        t for t in seq
        if t not in (sos_id, pad_id)
    ]

    if eos_id in seq:
        seq = seq[:seq.index(eos_id)]

    return seq

In [ ]:
def length_penalty(seq, alpha):
    return ((5 + len(seq)) / 6) ** alpha


def eos_adjust(
    score,
    seq,
    eos_id,
    eos_bonus=0.0,
    eos_penalty=0.0
):
    if seq[-1] == eos_id:
        return score + eos_bonus

    return score - eos_penalty

In [ ]:
def greedy_search(
    model,
    src,
    src_mask,
    sos_id,
    eos_id,
    pad_id=0,
    max_len=200
):

    device = src.device

    model.eval()

    src_key_padding_mask = (
        src_mask == 0
    ).to(device)

    with torch.no_grad():

        memory = model.encoder(
            src,
            src_key_padding_mask=src_key_padding_mask
        )

        seq = [sos_id]

        for _ in range(max_len):

            tgt = torch.tensor(
                seq,
                device=device
            ).unsqueeze(0)

            tgt_mask = model.generate_square_subsequent_mask(
                len(seq)
            ).to(device)

            tgt_key_padding_mask = (
                tgt == pad_id
            ).to(device)

            logits, _ = model.decoder(
                tgt,
                memory,
                tgt_mask,
                tgt_key_padding_mask,
                src_key_padding_mask
            )

            next_token = torch.argmax(
                logits[:, -1],
                dim=-1
            ).item()

            seq.append(next_token)

            if next_token == eos_id:
                break

    return seq

In [ ]:
def beam_search(
    model,
    src,
    src_mask,
    sos_id,
    eos_id,
    pad_id=0,
    beam_size=5,
    max_len=200,
    alpha=0.6,
    no_repeat_ngram=3,
    repeat_penalty=1.2
):

    device = src.device

    model.eval()

    src_key_padding_mask = (
        src_mask == 0
    ).to(device)

    with torch.no_grad():

        memory = model.encoder(
            src,
            src_key_padding_mask=src_key_padding_mask
        )

        beams = [
            ([sos_id], 0.0)
        ]

        for _ in range(max_len):

            candidates = []

            for seq, score in beams:

                if seq[-1] == eos_id:
                    candidates.append(
                        (seq, score)
                    )
                    continue

                tgt = torch.tensor(
                    seq,
                    device=device
                ).unsqueeze(0)

                tgt_mask = model.generate_square_subsequent_mask(
                    len(seq)
                ).to(device)

                tgt_key_padding_mask = (
                    tgt == pad_id
                )

                logits, _ = model.decoder(
                    tgt,
                    memory,
                    tgt_mask,
                    tgt_key_padding_mask,
                    src_key_padding_mask
                )

                log_probs = F.log_softmax(
                    logits[:, -1],
                    dim=-1
                ).squeeze(0)

                for tok in set(seq):
                    log_probs[tok] /= repeat_penalty

                if len(seq) >= no_repeat_ngram - 1:

                    prefix = seq[-(
                        no_repeat_ngram - 1
                    ):]

                    banned = set()

                    for i in range(
                        len(seq) - no_repeat_ngram + 1
                    ):
                        if seq[
                            i:i+no_repeat_ngram-1
                        ] == prefix:

                            banned.add(
                                seq[
                                    i+no_repeat_ngram-1
                                ]
                            )

                    for tok in banned:
                        log_probs[tok] = -1e9

                topk = torch.topk(
                    log_probs,
                    beam_size
                )

                for k in range(beam_size):

                    token = topk.indices[k].item()

                    token_score = topk.values[k].item()

                    candidates.append(
                        (
                            seq + [token],
                            score + token_score
                        )
                    )

            beams = sorted(
                candidates,
                key=lambda x:
                    x[1] /
                    (((5 + len(x[0])) / 6) ** alpha),
                reverse=True
            )[:beam_size]

            if all(
                s[-1] == eos_id
                for s, _ in beams
            ):
                break

    return beams[0][0]

In [ ]:
import heapq


def best_first_search(
    model,
    src,
    src_mask,
    sos_id,
    eos_id,
    pad_id=0,
    beam_size=5,
    max_len=200,
    alpha=0.6,
    no_repeat_ngram=3,
    repeat_penalty=1.2
):

    device = src.device

    model.eval()

    src_key_padding_mask = (
        src_mask == 0
    ).to(device)

    with torch.no_grad():

        memory = model.encoder(
            src,
            src_key_padding_mask=src_key_padding_mask
        )

        heap = [(0.0, [sos_id])]
        complete = []

        while heap:

            neg_score, seq = heapq.heappop(heap)

            score = -neg_score

            if (
                seq[-1] == eos_id
                or len(seq) >= max_len
            ):
                complete.append(
                    (seq, score)
                )

                if len(complete) >= beam_size:
                    break

                continue

            tgt = torch.tensor(
                seq,
                device=device
            ).unsqueeze(0)

            tgt_mask = model.generate_square_subsequent_mask(
                len(seq)
            ).to(device)

            tgt_key_padding_mask = (
                tgt == pad_id
            )

            logits, _ = model.decoder(
                tgt,
                memory,
                tgt_mask,
                tgt_key_padding_mask,
                src_key_padding_mask
            )

            log_probs = F.log_softmax(
                logits[:, -1],
                dim=-1
            ).squeeze(0)

            for tok in set(seq):
                log_probs[tok] /= repeat_penalty

            if len(seq) >= no_repeat_ngram - 1:

                prefix = seq[-(
                    no_repeat_ngram - 1
                ):]

                banned = set()

                for i in range(
                    len(seq) - no_repeat_ngram + 1
                ):
                    if seq[
                        i:i+no_repeat_ngram-1
                    ] == prefix:

                        banned.add(
                            seq[
                                i+no_repeat_ngram-1
                            ]
                        )

                for tok in banned:
                    log_probs[tok] = -1e9

            topk = torch.topk(
                log_probs,
                beam_size
            )

            for k in range(beam_size):

                token = topk.indices[k].item()

                token_score = topk.values[k].item()

                new_seq = seq + [token]

                new_score = score + token_score

                heapq.heappush(
                    heap,
                    (-new_score, new_seq)
                )

            if len(heap) > beam_size * 6:
                heap = heapq.nsmallest(
                    beam_size * 3,
                    heap
                )
                heapq.heapify(heap)

        if complete:

            complete = sorted(
                complete,
                key=lambda x:
                    x[1] /
                    (((5 + len(x[0])) / 6) ** alpha),
                reverse=True
            )

            return complete[0][0]

        return heapq.heappop(heap)[1]

In [ ]:
def greedy_decode(model, src, src_mask):
    return greedy_search(
        model,
        src,
        src_mask,
        sos_id,
        eos_id,
        pad_id
    )


def beam_decode(model, src, src_mask):
    return beam_search(
        model,
        src,
        src_mask,
        sos_id,
        eos_id,
        pad_id
    )


def best_first_decode(model, src, src_mask):
    return best_first_search(
        model,
        src,
        src_mask,
        sos_id,
        eos_id,
        pad_id
    )

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    epoch,
    sample_prob=0.1
):

    model.train()
    total_loss = 0

    for src, src_mask, tgt in loader:

        src = src.to(device)
        src_mask = src_mask.to(device)
        tgt = tgt.to(device)

        # safety
        src = torch.nan_to_num(src, 0.0)
        tgt = torch.nan_to_num(tgt, 0.0)

        # augmentation
        if np.random.rand() < 0.3:

            t0 = np.random.randint(
                0, src.shape[1] // 2
            )

            width = np.random.randint(5, 15)

            src[:, t0:t0+width, :] *= 0.1

        optimizer.zero_grad()

        decoder_input = tgt[:, :-1].clone()
        target = tgt[:, 1:]

        # teacher forcing noise
        mask = (
            torch.rand(
                decoder_input.shape,
                device=device
            ) < sample_prob
        )

        decoder_input[:, 1:] = torch.where(
            mask[:, 1:],
            decoder_input[:, :-1],
            decoder_input[:, 1:]
        )

        logits, cross_attn = model(
            src,
            src_mask,
            decoder_input
        )

        logits = torch.clamp(logits, -50, 50)

        logits_flat = logits.reshape(
            -1,
            logits.size(-1)
        )

        target_flat = target.reshape(-1)

        valid = target_flat != pad_id

        if valid.sum() == 0:
            continue

        ce_loss = criterion(
            logits_flat[valid],
            target_flat[valid]
        )

        if cross_attn is not None:
            cov_loss = compute_coverage_loss(
                cross_attn
            )
        else:
            cov_loss = 0.0

        loss = ce_loss + 0.05 * cov_loss

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        update_ema(model, ema_model, ema_decay)

        total_loss += loss.item()

    return total_loss / max(1, len(loader))

In [ ]:
def get_lr(
    epoch,
    base_lr=3e-4,
    warmup_epochs=3,
    max_epochs=100
):

    if epoch < warmup_epochs:

        return base_lr * (
            epoch / warmup_epochs
        )

    progress = (
        epoch - warmup_epochs
    ) / (max_epochs - warmup_epochs)

    return base_lr * 0.5 * (
        1 + math.cos(math.pi * progress)
    )


def update_ema(model, ema_model, decay=0.999):

    with torch.no_grad():

        for p, ema_p in zip(
            model.parameters(),
            ema_model.parameters()
        ):
            ema_p.data.mul_(decay).add_(
                p.data,
                alpha=1 - decay
            )

In [ ]:
def token_accuracy(output, trg, pad_idx=0):

    pred = output.argmax(1)

    mask = trg != pad_idx

    correct = (pred == trg) & mask

    return (
        correct.sum().item()
        / mask.sum().item()
    ) * 100


def clean(seq, sos_id, eos_id, pad_id):

    seq = [
        t for t in seq
        if t not in (sos_id, pad_id)
    ]

    if eos_id in seq:
        seq = seq[:seq.index(eos_id)]

    return seq

In [ ]:
def compute_coverage_loss(cross_attn_layers):

    attn = cross_attn_layers[-1]

    attn = attn.mean(dim=1)

    batch, tgt_len, src_len = attn.shape

    coverage = torch.zeros(
        batch,
        src_len,
        device=attn.device
    )

    cov_loss = 0.0

    for t in range(tgt_len):

        a_t = attn[:, t, :]

        cov_loss += torch.sum(
            torch.min(a_t, coverage)
        )

        coverage = coverage + a_t

    return cov_loss / batch

In [ ]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        input_dim,
        vocab_size,
        hidden_dim=256
    ):

        super().__init__()

        self.encoder = EncoderTransformer(
            input_dim,
            hidden_dim
        )

        self.decoder = CustomTransformerDecoder(
            vocab_size,
            hidden_dim
        )

    def generate_square_subsequent_mask(self, size):

        return torch.triu(
            torch.ones(
                size,
                size,
                device=next(
                    self.parameters()
                ).device
            ),
            diagonal=1
        ).bool()

    def forward(self, src, src_mask, tgt):

        src = src.to(device)
        tgt = tgt.to(device)
        src_mask = src_mask.to(device)

        src_key_padding_mask = (
            src_mask == 0
        ).bool()

        tgt_key_padding_mask = (
            tgt == pad_id
        ).bool()

        memory = self.encoder(
            src,
            src_key_padding_mask=src_key_padding_mask
        )

        tgt_mask = self.generate_square_subsequent_mask(
            tgt.size(1)
        )

        logits, cross_attn = self.decoder(
            tgt,
            memory,
            tgt_mask,
            tgt_key_padding_mask,
            src_key_padding_mask
        )

        return logits, cross_attn

In [ ]:
num_features = X_tensor.shape[2]
hidden_size = 256

model = Seq2Seq(
    input_dim=num_features,
    vocab_size=vocab_size,
    hidden_dim=hidden_size
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

ema_model = copy.deepcopy(model)
ema_decay = 0.999

In [ ]:
num_epochs = 100
patience = 50

best_val_loss = float("inf")
best_cer = float("inf")
patience_counter = 0

os.makedirs("checkpoints", exist_ok=True)

print("Starting training...")

for epoch in range(num_epochs):

    print(f"\nEpoch {epoch+1}/{num_epochs}")

    lr = get_lr(epoch)

    for pg in optimizer.param_groups:
        pg["lr"] = lr

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        epoch
    )

    print("Train Loss:", train_loss)

    eval_model = ema_model
    eval_model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for src, src_mask, tgt in val_loader:

            src = src.to(device)
            tgt = tgt.to(device)

            logits, _ = ema_model(
                src,
                src_mask,
                tgt[:, :-1]
            )

            pred = logits.reshape(
                -1,
                logits.size(-1)
            )

            gold = tgt[:, 1:].reshape(-1)

            mask = gold != pad_id

            loss = criterion(
                pred[mask],
                gold[mask]
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print("Val Loss:", val_loss)

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            model.state_dict(),
            "checkpoints/best.pt"
        )

    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping")
        break

torch.save(
    model.state_dict(),
    "checkpoints/last.pt"
)

Starting training...

Epoch 1/100


Train Loss: 12.042249136359073


/home/ubuntu/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Val Loss: 5.405268504701811

Epoch 2/100
Train Loss: 10.025640160930234
Val Loss: 4.151432604625307

Epoch 3/100
Train Loss: 9.309123702937582
Val Loss: 3.4212397542493096

Epoch 4/100
Train Loss: 9.113603689941145
Val Loss: 3.160687808332772

Epoch 5/100
Train Loss: 8.907886164091384
Val Loss: 2.9654261408181024

Epoch 6/100
Train Loss: 8.811603678880973
Val Loss: 2.826706894512834

Epoch 7/100
Train Loss: 8.71317200385018
Val Loss: 2.7719259673151475

Epoch 8/100


In [ ]:
def average_checkpoints(paths):

    avg = None

    for p in paths:

        state = torch.load(p)

        if avg is None:

            avg = state

            for k in avg:
                avg[k] = avg[k].float()

        else:

            for k in avg:
                avg[k] += state[k].float()

    for k in avg:
        avg[k] /= len(paths)

    return avg

In [ ]:
paths = [
    "checkpoints/model_epoch_70.pt",
    "checkpoints/model_epoch_80.pt",
    "checkpoints/model_epoch_90.pt"
]

avg_state = average_checkpoints(paths)

model.load_state_dict(avg_state)
ema_model.load_state_dict(avg_state)

model.eval()
ema_model.eval()

In [ ]:
def decode_and_clean(seq, id_to_token):

    seq = clean(seq, sos_id, eos_id, pad_id)

    return [id_to_token[t] for t in seq]

In [ ]:
from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

smooth = SmoothingFunction().method1


def compute_bleu(pred, true):

    return {
        "BLEU-1": sentence_bleu(
            [true],
            pred,
            weights=(1,0,0,0),
            smoothing_function=smooth
        ),

        "BLEU-2": sentence_bleu(
            [true],
            pred,
            weights=(0.5,0.5,0,0),
            smoothing_function=smooth
        ),

        "BLEU-4": sentence_bleu(
            [true],
            pred,
            weights=(0.25,0.25,0.25,0.25),
            smoothing_function=smooth
        ),
    }


def compute_cer(pred, true):

    pred_str = "".join(pred)
    true_str = "".join(true)

    if len(true_str) == 0:
        return 1.0 if len(pred_str) > 0 else 0.0

    return Lev.distance(
        pred_str,
        true_str
    ) / len(true_str)

In [ ]:
def compute_metrics(preds, trues):

    bleu_scores = []
    cer_scores = []
    exact = 0

    for pred, true in zip(preds, trues):

        bleu_scores.append(
            compute_bleu(pred, true)
        )

        cer_scores.append(
            compute_cer(pred, true)
        )

        if pred == true:
            exact += 1

    return {
        "BLEU-1": np.mean(
            [b["BLEU-1"] for b in bleu_scores]
        ),

        "BLEU-2": np.mean(
            [b["BLEU-2"] for b in bleu_scores]
        ),

        "BLEU-4": np.mean(
            [b["BLEU-4"] for b in bleu_scores]
        ),

        "CER": np.mean(cer_scores),

        "EXACT": exact / len(preds)
    }

In [ ]:
def evaluate(model, loader, decoding_fn):

    model.eval()

    preds = []
    trues = []

    with torch.no_grad():

        for src, src_mask, tgt in loader:

            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):

                pred_seq = decoding_fn(
                    model,
                    src[i].unsqueeze(0),
                    src_mask[i].unsqueeze(0)
                )

                true_seq = tgt[i].tolist()

                pred_seq = clean(
                    pred_seq,
                    sos_id,
                    eos_id,
                    pad_id
                )

                true_seq = clean(
                    true_seq,
                    sos_id,
                    eos_id,
                    pad_id
                )

                preds.append(pred_seq)
                trues.append(true_seq)

    decoded_preds = [
        [id_to_token[t] for t in p]
        for p in preds
    ]

    decoded_trues = [
        [id_to_token[t] for t in t_]
        for t_ in trues
    ]

    return compute_metrics(
        decoded_preds,
        decoded_trues
    )

In [ ]:
greedy_results = evaluate(
    ema_model,
    val_loader,
    greedy_decode
)

print("GREEDY:", greedy_results)


beam_results = evaluate(
    ema_model,
    val_loader,
    beam_decode
)

print("BEAM:", beam_results)


best_first_results = evaluate(
    ema_model,
    val_loader,
    best_first_decode
)

print("BEST-FIRST:", best_first_results)

In [ ]:
def show_examples_all_decoders(
    model,
    loader,
    n=10
):

    model.eval()

    decoders = {
        "GREEDY": greedy_decode,
        "BEAM": beam_decode,
        "BEST-FIRST": best_first_decode
    }

    examples = []

    with torch.no_grad():

        for src, src_mask, tgt in loader:

            src = src.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):

                true_ids = clean(
                    tgt[i].tolist(),
                    sos_id,
                    eos_id,
                    pad_id
                )

                true_str = "".join(
                    id_to_token[t]
                    for t in true_ids
                )

                preds = {}

                for name, fn in decoders.items():

                    pred_ids = fn(
                        model,
                        src[i].unsqueeze(0),
                        src_mask[i].unsqueeze(0)
                    )

                    pred_ids = clean(
                        pred_ids,
                        sos_id,
                        eos_id,
                        pad_id
                    )

                    pred_str = "".join(
                        id_to_token[t]
                        for t in pred_ids
                    )

                    preds[name] = pred_str

                examples.append((true_str, preds))

                if len(examples) >= n:
                    return examples

    return examples

In [ ]:
model.load_state_dict(
    torch.load(
        "checkpoints/best_sequence_model.pt"
    )
)

model.eval()

examples = show_examples_all_decoders(
    ema_model,
    val_loader,
    n=10
)

for i, (true, preds) in enumerate(examples):

    print(f"\nExample {i+1}")

    print("TRUE:       ", true)
    print("GREEDY:     ", preds["GREEDY"])
    print("BEAM:       ", preds["BEAM"])
    print("BEST-FIRST: ", preds["BEST-FIRST"])

In [ ]:
with torch.no_grad():

    src1 = X_val_tensor[0:1].to(device)
    src2 = X_val_tensor[1:2].to(device)

    mem1 = model.encoder(src1)
    mem2 = model.encoder(src2)

    diff = (mem1 - mem2).abs().mean()

    print("Encoder difference:", diff.item())

In [ ]:
with torch.no_grad():

    tgt = torch.tensor(
        [[sos_id]]
    ).to(device)

    tgt_mask = model.generate_square_subsequent_mask(
        tgt.size(1)
    ).to(device)

    logits, attn = model.decoder(
        tgt=tgt,
        memory=mem1,
        tgt_mask=tgt_mask,
        tgt_key_padding_mask=None,
        memory_key_padding_mask=None
    )

    a = attn[-1]

    print("Attention mean:", a.mean().item())
    print("Attention std:", a.std().item())

In [ ]:
with torch.no_grad():

    src1 = X_val_tensor[0:1].to(device)
    src2 = X_val_tensor[1:2].to(device)

    p1 = beam_search(
        model,
        src1,
        sos_id,
        eos_id,
        pad_id
    )

    p2 = beam_search(
        model,
        src2,
        sos_id,
        eos_id,
        pad_id
    )

    print("P1 == P2?", p1 == p2)

In [ ]:
import shap

model.eval()

class WrappedModel(torch.nn.Module):

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):

        batch = x.shape[0]

        tgt = torch.tensor(
            [[sos_id]] * batch
        ).to(x.device)

        src_mask = torch.ones(
            x.shape[:2],
            dtype=torch.bool
        ).to(x.device)

        logits, _ = self.model(
            x,
            src_mask,
            tgt
        )

        probs = torch.softmax(
            logits[:, -1, :],
            dim=-1
        )

        return probs[:, 10]

In [ ]:
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "cer_greedy": [],
    "cer_beam": [],
    "bleu": [],
}

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(history["epoch"], history["train_loss"], label="Train Loss")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss")
plt.plot(history["epoch"], history["cer_beam"], label="CER Beam")
plt.plot(history["epoch"], history["bleu"], label="BLEU")

plt.legend()
plt.title("Training Progress Overview")
plt.xlabel("Epoch")

plt.show()

In [ ]:
def encoder_stats(model, loader):

    model.eval()

    all_means = []
    all_stds = []

    with torch.no_grad():

        for src, src_mask, _ in loader:

            src = src.to(device)

            mem = model.encoder(src)

            all_means.append(
                mem.mean().item()
            )

            all_stds.append(
                mem.std().item()
            )

    return np.mean(all_means), np.mean(all_stds)

In [ ]:
plt.plot(epoch_list, mean_list, label="Mean activation")
plt.plot(epoch_list, std_list, label="Std activation")

plt.legend()
plt.title("Encoder Representation Drift")
plt.show()

In [ ]:
from sklearn.manifold import TSNE

def plot_embedding_space(model, loader, n_batches=5):

    model.eval()

    feats = []

    with torch.no_grad():

        for i, (src, src_mask, _) in enumerate(loader):

            if i >= n_batches:
                break

            src = src.to(device)

            mem = model.encoder(src)

            feats.append(
                mem.mean(dim=1).cpu().numpy()
            )

    feats = np.concatenate(feats, axis=0)

    tsne = TSNE(n_components=2)

    proj = tsne.fit_transform(feats)

    plt.scatter(proj[:,0], proj[:,1])
    plt.title("Encoder Space Structure")
    plt.show()

In [ ]:
def attention_entropy(attn):

    p = attn.mean(dim=1)

    p = p + 1e-8

    entropy = -(p * torch.log(p)).sum(dim=-1)

    return entropy.mean().item()

In [ ]:
from collections import Counter

def token_distribution(model, loader):

    model.eval()

    counter = Counter()

    with torch.no_grad():

        for src, src_mask, _ in loader:

            src = src.to(device)

            logits, _ = model(
                src,
                src_mask,
                tgt=torch.tensor([[sos_id]]).to(device)
            )

            preds = logits.argmax(-1).flatten().tolist()

            counter.update(preds)

    return counter

In [ ]:
def motion_energy(seq):

    return np.mean(
        np.linalg.norm(
            seq[1:] - seq[:-1],
            axis=1
        )
    )

In [ ]:
importance_per_frame = vals.reshape(
    num_frames,
    -1
).mean(axis=1)

plt.plot(importance_per_frame)
plt.title("Which frames matter most")
plt.show()

In [ ]:
"""
Key observations:

- BLEU is computed using NLTK sentence_bleu
- CER uses Levenshtein distance normalized by length
- EXACT match is strict sequence equality
- Beam search improves accuracy but reduces diversity
- Encoder representation drift should stay stable
- Attention entropy can indicate overconfidence collapse

Potential improvements:

1. Relative positional encoding
2. Stronger data augmentation (temporal warp)
3. Label smoothing tuning
4. Better coverage penalty scheduling
5. Nucleus sampling decoding
6. Multi-signer conditioning (reduce bias)
"""